<a href="https://colab.research.google.com/github/tmzt/TrainingExperiments/blob/main/Highbay/Local/HighbaySchemaProseFinetune4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Highbay schema/prose fine-tune - v4

Sentence in, `GeniusAst` JSON out. Compact by design: the rationale for every
choice lives in `Highbay/Local/data/README.md` and in v3's history, not in this
notebook.

Two things v3 lacked.

**Invariants are named, checked and recorded.** Every `check(...)` below is a
gate that halts the run and lands in the results file. v3 asserted in places and
printed in others, so a run's trustworthiness had to be reconstructed from
scrollback.

**The run emits a complete results artifact**, not a screenful. v3's eval
printed `0/28` and there was no way to tell a broken metric from a broken model
without re-running by hand - it was the metric, and `results.json` now contains
every prediction beside its target so that question is answerable after the
fact.

**The metric v3 got wrong:** it compared `canonical(ast)` verbatim, and
`ui_prompt` is a generated SENTENCE inside that AST. Requiring prose word-for-word
made exact-match unreachable even for a perfect structural answer, and dragged
both eval and the memorization check to zero. v4 scores STRUCTURE (the AST minus
`ui_prompt`) as the headline and reports full match separately.

Change `BASE_MODEL` to move between sizes - a run is ~2 minutes on a T4, so the
base is a cheap variable, not a commitment.

In [ ]:
# 1. Config. Everything tunable is here.
# Qwen2.5-1.5B, chosen against Llama-3.2-1B for two MEASURED reasons:
#
#  1. THE EMBEDDING TENSOR FITS AND LLAMA'S DOES NOT. Mobile WebGPU reports a
#     128 MiB maxStorageBufferBindingSize (our Adreno 740 does; only ~1/3 of
#     Android exceeds the floor), and the largest tensor is token_embd. Qwen's
#     1.5B is 151936x1536 = 125 MiB at q4_K; Llama-3.2-1B is 128256x2048 =
#     141 MiB and clears no sensible quantization. The LARGER model is the one
#     that fits, because vocab x hidden - not parameter count - is the bound.
#  2. QWEN2.5 IS NATIVELY CHATML. <|im_start|>=151644 and <|im_end|>=151645 are
#     in its vocabulary and its stock template emits them, so the trained
#     template and the one baked into an exported GGUF are the same. On
#     Llama-3.2 the get_chat_template(..., "chatml") call below is a silent
#     remap to <|eot_id|>, and the served prompt would not match training.
BASE_MODEL   = "unsloth/Qwen2.5-1.5B-Instruct"
RUN_ID       = "v4-qwen2.5-1.5b"
SEED         = 3407
EPOCHS       = 3
LR           = 2e-4
LORA_R       = 16
EVAL_FRACTION = 0.2
TARGET       = "ast"            # "ast" | "ui_prompt"
MAX_SEQ_LENGTH = 2048

REPO_URL  = "https://github.com/tmzt/TrainingExperiments.git"
CHECKOUT  = "/content/TrainingExperiments"
DATA_DIR  = f"{CHECKOUT}/Highbay/Local/data"

import json, os, random, collections, subprocess, hashlib
from datetime import datetime, timezone

# ONE timestamp, taken at the top and used for every path below. A run is a
# directory, and re-running the save cell writes back into the SAME directory
# rather than minting a second one - only re-running this cell starts a new run.
# Previously the destination was RUN_ID alone, so a rerun silently replaced the
# last result; naming by RUN_ID + start time makes runs accumulate and stay
# comparable, which is the point of keeping results.json at all.
RUN_STARTED = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR     = f"{RUN_ID}-{RUN_STARTED}"
OUT_DIR     = f"/content/{RUN_DIR}"        # everything this run produces
print(f"run {RUN_DIR}")

INVARIANTS = []
def check(name, ok, detail=""):
    """A gate. Halts the run AND lands in results.json, so a run's
    trustworthiness is data rather than something reconstructed from scrollback."""
    INVARIANTS.append({"name": name, "ok": bool(ok), "detail": str(detail)})
    print(f"  [{'ok  ' if ok else 'FAIL'}] {name}" + (f" - {detail}" if detail else ""))
    if not ok:
        raise AssertionError(f"{name}: {detail}")
    return ok

In [ ]:
# 2. Hardware. dtype from CAPABILITY - torch.cuda.is_bf16_supported() counts
#    emulation, answers True on a T4, and TrainingArguments then refuses.
import torch

GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
check("gpu present", torch.cuda.is_available(), GPU_NAME)

MAJOR, MINOR = torch.cuda.get_device_capability()
BF16    = MAJOR >= 8
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
BATCH   = 2 if VRAM_GB < 24 else (4 if VRAM_GB < 48 else 8)
ACCUM   = max(1, 8 // BATCH)

print(f"  {GPU_NAME}  capability {MAJOR}.{MINOR}  {VRAM_GB:.0f} GB")
print(f"  dtype {'bf16' if BF16 else 'fp16'}   batch {BATCH} x accum {ACCUM}")

In [ ]:
# 3. Google Drive, FIRST - before the GPU work, not after it.
#
# v3 mounted at the very end, so a failed mount came after training had already
# run and the adapter existed only on a runtime about to be recycled. Mounting
# here means a bad mount costs five seconds instead of a whole run, and you get
# the auth prompt while you are still watching.
from google.colab import drive

DRIVE_ROOT = "/content/drive"
DRIVE_DEST = f"{DRIVE_ROOT}/MyDrive/Training Data/{RUN_ID}"

mounted = False
for attempt in (1, 2, 3):
    try:
        # force_remount from the second attempt: a half-finished mount left by a
        # dismissed auth popup is the usual cause of "ValueError: mount failed",
        # and a plain retry hits the same stale state.
        drive.mount(DRIVE_ROOT, force_remount=(attempt > 1))
        mounted = os.path.isdir(f"{DRIVE_ROOT}/MyDrive")
        if mounted:
            break
    except Exception as e:
        print(f"  mount attempt {attempt} failed: {type(e).__name__}: {e}")

# Mounted is not the same as writable - quota and permission problems only show
# up on write, and we would rather find out now than after training.
writable = False
if mounted:
    try:
        os.makedirs(DRIVE_DEST, exist_ok=True)
        probe = f"{DRIVE_DEST}/.write_probe"
        with open(probe, "w") as fh:
            fh.write("ok")
        os.remove(probe)
        writable = True
    except Exception as e:
        print(f"  mounted but NOT writable: {type(e).__name__}: {e}")

check("Drive mounted and writable", writable, DRIVE_DEST)

In [ ]:
# 4. Dependencies - RESOLVED. --no-deps keeps whatever the image ships, which is
#    how trl<0.9.0 ended up beside a transformers that rejects tokenizer=.
TORCH = torch.__version__.split("+")[0]
!pip install -q --upgrade unsloth unsloth_zoo "torch=={TORCH}"

In [ ]:
# 5. What landed, and is it self-consistent?
import importlib.metadata as md_, inspect
from transformers import Trainer

VERSIONS = {}
for pkg in ("torch","transformers","peft","accelerate","bitsandbytes","datasets",
            "trl","unsloth","unsloth_zoo"):
    try: VERSIONS[pkg] = md_.version(pkg)
    except Exception: VERSIONS[pkg] = None
print("  " + "  ".join(f"{k}={v}" for k, v in VERSIONS.items() if v))

check("torch still sees the GPU after install", torch.cuda.is_available(),
      "the resolver replaced torch - delete the runtime and rerun")
params = inspect.signature(Trainer.__init__).parameters
check("Trainer signature recognised",
      "processing_class" in params or "tokenizer" in params,
      f"processing_class={'processing_class' in params} tokenizer={'tokenizer' in params}")

In [ ]:
# 6. Corpus. Already merged, deduped and normalized by clean_genius_corpus.py.
if not os.path.isdir(DATA_DIR):
    !git clone --depth 1 $REPO_URL $CHECKOUT

CORPUS = f"{DATA_DIR}/genius_corpus_clean.jsonl"
raw = open(CORPUS, "rb").read()
CORPUS_SHA = hashlib.sha256(raw).hexdigest()[:16]
records = [json.loads(l) for l in raw.decode("utf-8").splitlines() if l.strip()]

INTENT_TYPES  = {"PIPELINE", "SCHEMA_SUGGESTION"}
TRIGGER_TYPES = {"ON_CREATE", "ON_UPDATE", "ON_DELETE", "SCHEDULED"}
ACTION_TYPES  = {"SEND_NOTIFICATION", "UPDATE_RECORD", "CALCULATE"}
COLUMN_TYPES  = {"NUMBER", "DATE", "STRING", "BOOLEAN", "RELATION"}
MUTATION_KEYS = {"CREATE_TABLE": {"action","name"},
                 "ADD_COLUMN":   {"action","table","column_name","type"}}

def canonical(ast):
    return json.dumps(ast, sort_keys=True, separators=(",", ":"))

def structure(ast):
    """The AST minus ui_prompt. ui_prompt is a generated SENTENCE; requiring it
    verbatim is what made v3's exact-match unreachable and reported 0/10 on the
    TRAINING split, which no memorizing model can actually score."""
    return {k: v for k, v in ast.items() if k != "ui_prompt"} if isinstance(ast, dict) else ast

def validate(ast):
    p = []
    if not isinstance(ast, dict): return ["not an object"]
    it = ast.get("intent_type")
    if it not in INTENT_TYPES: p.append(f"intent_type={it!r}")
    if it == "PIPELINE":
        pa = ast.get("pipeline_ast")
        if not isinstance(pa, dict): p.append("PIPELINE without pipeline_ast")
        else:
            t, a = pa.get("trigger"), pa.get("action")
            if not isinstance(t, dict): p.append("trigger missing")
            elif t.get("type") not in TRIGGER_TYPES: p.append(f"trigger.type={t.get('type')!r}")
            if not isinstance(a, dict): p.append("action missing")
            elif a.get("type") not in ACTION_TYPES: p.append(f"action.type={a.get('type')!r}")
    if it == "SCHEMA_SUGGESTION":
        muts = ast.get("schema_mutations")
        if not isinstance(muts, list) or not muts: p.append("no schema_mutations")
        else:
            for i, m in enumerate(muts):
                want = MUTATION_KEYS.get(m.get("action"))
                if want is None: p.append(f"mutation[{i}].action={m.get('action')!r}")
                elif set(m) != want: p.append(f"mutation[{i}] keys {sorted(set(m))}")
                elif m["action"] == "ADD_COLUMN" and m.get("type") not in COLUMN_TYPES:
                    p.append(f"mutation[{i}].type={m.get('type')!r}")
    return p

bad = [(i, p) for i, p in ((i, validate(r["output_ast"])) for i, r in enumerate(records)) if p]
check("corpus validates", not bad, f"{len(records)} records, sha {CORPUS_SHA}, {len(bad)} bad")
INTENTS = dict(collections.Counter(r["output_ast"]["intent_type"] for r in records))
print(f"  {INTENTS}")

In [ ]:
# 7. Model + LoRA
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL, max_seq_length = MAX_SEQ_LENGTH, load_in_4bit = True)
model = FastLanguageModel.get_peft_model(
    model, r = LORA_R, lora_alpha = LORA_R, lora_dropout = 0, bias = "none",
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth", random_state = SEED)
tokenizer = get_chat_template(tokenizer, chat_template = "chatml")

TRAINABLE = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  {TRAINABLE:,} trainable  ->  adapter ~{TRAINABLE*2/1024**2:.0f} MiB at fp16")

In [ ]:
# 8. ChatML, split, and the length limits DERIVED from the corpus.
SYSTEM = ("You convert a user's plain-language automation request into a strict "
          "JSON AST. Reply with JSON only - no prose, no code fences.")
SYSTEM_PROSE = ("You restate a user's plain-language automation request as a "
                "short confirming question. Reply with one sentence.")

def system_for(): return SYSTEM_PROSE if TARGET == "ui_prompt" else SYSTEM
def answer_for(r):
    return r["output_ast"].get("ui_prompt","") if TARGET == "ui_prompt" else canonical(r["output_ast"])
def messages(r, with_answer=True):
    m = [{"role":"system","content":system_for()},
         {"role":"user","content":r["user_input"]}]
    return m + ([{"role":"assistant","content":answer_for(r)}] if with_answer else m[:0])
def to_text(r):
    return tokenizer.apply_chat_template(messages(r), tokenize=False, add_generation_prompt=False)
def prompt_for(user_input):
    return tokenizer.apply_chat_template(
        [{"role":"system","content":system_for()},{"role":"user","content":user_input}],
        tokenize=False, add_generation_prompt=True)

if TARGET == "ast":
    check("ASTs survive ChatML byte-identical",
          all(canonical(json.loads(answer_for(r))) == canonical(r["output_ast"]) for r in records),
          f"{len(records)} records")

by_intent = collections.defaultdict(list)
for r in records: by_intent[r["output_ast"]["intent_type"]].append(r)
train_recs, eval_recs, rng = [], [], random.Random(SEED)
for _, g in sorted(by_intent.items()):
    g = g[:]; rng.shuffle(g)
    cut = max(1, round(len(g) * EVAL_FRACTION))
    eval_recs += g[:cut]; train_recs += g[cut:]
rng.shuffle(train_recs); rng.shuffle(eval_recs)

full_lens   = [len(tokenizer(to_text(r), add_special_tokens=False)["input_ids"]) for r in records]
target_lens = [len(tokenizer(answer_for(r), add_special_tokens=False)["input_ids"]) for r in records]
LONGEST_FULL, LONGEST_TARGET = max(full_lens), max(target_lens)
# 2x + 64, not 1.5x + 32: at 1.5x one eval output ran out of room mid-JSON, and a
# truncated generation is a miss for a reason that is not the model's.
MAX_NEW_TOKENS = LONGEST_TARGET * 2 + 64

check("no example is truncated", LONGEST_FULL <= MAX_SEQ_LENGTH,
      f"longest {LONGEST_FULL} <= {MAX_SEQ_LENGTH}")
print(f"  train {len(train_recs)}  eval {len(eval_recs)}  "
      f"longest target {LONGEST_TARGET} -> MAX_NEW_TOKENS {MAX_NEW_TOKENS}")

In [ ]:
# 9. Eval. Reports STRUCTURE (headline) and FULL (stricter) separately, and
#    keeps every prediction so a bad number can be diagnosed without a rerun.
def parse_ast(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        text = text[4:] if text.lower().startswith("json") else text
    s = text.find("{")
    if s < 0: return None
    d = 0
    for i, ch in enumerate(text[s:], s):
        d += (ch == "{") - (ch == "}")
        if d == 0:
            try: return json.loads(text[s:i+1])
            except json.JSONDecodeError: return None
    return None

@torch.no_grad()
def predict(user_input):
    ids = tokenizer(prompt_for(user_input), return_tensors="pt",
                    add_special_tokens=False).to(model.device)
    out = model.generate(**ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def diff_fields(got, want):
    if not isinstance(got, dict): return ["<unparsed>"]
    return sorted(k for k in set(got) | set(want)
                  if canonical(got.get(k)) != canonical(want.get(k)))

def decisions(got, want):
    """Per-DECISION scoring - the metric that actually carries signal.

    Measured on the spike (Qwen2.5-0.5B, Metal, grammar on): exact-structure
    match scored 0/20 for EVERY prompt variant including 6-shot, because it
    demands the corpus author's exact table name (`Water_Logs`, not
    `water_logs`), exact column names and exact payload prose. Nothing scores on
    that. Per-decision separated the arms cleanly where structure could not -
    n_mutations went 2/7 -> 5/7 while structure stayed flat at zero.
    """
    out = {"intent": got.get("intent_type") == want.get("intent_type")}
    pg, pw = got.get("pipeline_ast") or {}, want.get("pipeline_ast") or {}
    if pw:
        out["trigger"] = pg.get("trigger", {}).get("type") == pw.get("trigger", {}).get("type")
        out["action"] = pg.get("action", {}).get("type") == pw.get("action", {}).get("type")
    mw = want.get("schema_mutations") or []
    if mw:
        mg = got.get("schema_mutations") or []
        out["n_mutations"] = len(mg) == len(mw)
        out["mutation_kinds"] = ([m.get("action") for m in mg]
                                 == [m.get("action") for m in mw])
        out["col_types"] = (sorted(m.get("type") or "" for m in mg)
                            == sorted(m.get("type") or "" for m in mw))
    # Names are free-form, so compare case/underscore-insensitively: "close" is
    # meaningful for an invented identifier where it is not for an enum.
    norm = lambda x: (x or "").lower().replace("_", "")
    first = lambda ms: (ms[0].get("table") or ms[0].get("name")) if ms else None
    tw = norm((pw.get("trigger") or {}).get("table")) or norm(first(mw))
    tg = norm((pg.get("trigger") or {}).get("table")) or norm(
        first(got.get("schema_mutations") or []))
    if tw:
        out["table_name"] = tg == tw
    return out


def evaluate(dataset, label, phase):
    rows, m = [], collections.Counter()
    hits, seen = collections.Counter(), collections.Counter()
    for r in dataset:
        raw = predict(r["user_input"])
        got, want = parse_ast(raw), r["output_ast"]
        row = {"phase": phase, "user_input": r["user_input"],
               "target": want, "raw": raw, "parsed": got}
        if got is None:
            row["unclosed"] = raw.count("{") > raw.count("}")
            m["unclosed"] += row["unclosed"]
        else:
            m["parseable"] += 1
            m["valid"]      += not validate(got)
            m["structure"]  += canonical(structure(got)) == canonical(structure(want))
            m["full"]       += canonical(got) == canonical(want)
            m["intent"]     += got.get("intent_type") == want.get("intent_type")
            row["diff_fields"] = diff_fields(got, want)
        row["decisions"] = decisions(got, want)
        for k, v in row["decisions"].items():
            seen[k] += 1
            hits[k] += bool(v)
        rows.append(row)
    n = len(dataset)
    print(f"--- {label} (n={n}) ---")
    for k in ("parseable","valid","structure","full","intent"):
        star = "   <- the number that matters" if k == "structure" else ""
        print(f"  {k:10} {m[k]}/{n}{star}")
    if m["unclosed"]:
        print(f"  !! {m['unclosed']} unclosed - raise MAX_NEW_TOKENS ({MAX_NEW_TOKENS}); not model errors")
    # The per-decision line, with chance printed beside it - intent is a 2-way
    # choice, so 50% is noise, not a result.
    CHANCE = {"intent": 0.50, "trigger": 0.25, "action": 0.33}
    print("  decisions:")
    for k in ("intent", "trigger", "action", "n_mutations", "mutation_kinds",
              "col_types", "table_name"):
        if seen[k]:
            c = f"   (chance {CHANCE[k]:.0%})" if k in CHANCE else ""
            print(f"    {k:15} {hits[k]}/{seen[k]}  {hits[k]/seen[k]:5.0%}{c}")
    # WHICH fields break a near-miss is the actionable part
    fields = collections.Counter(f for r in rows for f in r.get("diff_fields", []))
    if fields: print(f"  fields differing: {dict(fields)}")
    return {"n": n,
            **{k: m[k] for k in ("parseable","valid","structure","full","intent","unclosed")},
            "decisions": {k: {"hit": hits[k], "seen": seen[k]} for k in seen}}, rows

FastLanguageModel.for_inference(model)
model.generation_config.max_length = None      # it carries the base's 131072
RESULTS, PREDICTIONS = {}, []
RESULTS["baseline"], rows = evaluate(eval_recs, "BASELINE", "baseline"); PREDICTIONS += rows

In [ ]:
# 10. Train
from transformers import Trainer, TrainingArguments
from datasets import Dataset

FastLanguageModel.for_training(model)
model.config.use_cache = False
BF16 = torch.cuda.get_device_capability()[0] >= 8      # never from is_bf16_supported()

ASSIST = tokenizer("<|im_start|>assistant\n", add_special_tokens=False)["input_ids"]
def encode(r):
    ids = tokenizer(to_text(r), truncation=True, max_length=MAX_SEQ_LENGTH,
                    add_special_tokens=False)["input_ids"]
    cut = 0
    for i in range(len(ids) - len(ASSIST), -1, -1):
        if ids[i:i+len(ASSIST)] == ASSIST: cut = i + len(ASSIST); break
    return {"input_ids": ids, "labels": [-100]*cut + ids[cut:]}

train_ds, eval_ds = Dataset.from_list([encode(r) for r in train_recs]), \
                    Dataset.from_list([encode(r) for r in eval_recs])
scored = sum(l != -100 for l in train_ds[0]["labels"]); total = len(train_ds[0]["input_ids"])
check("loss is masked to the answer", 0 < scored < total,
      f"{scored}/{total} scored ({scored/total:.0%})")

pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
def collate(b):
    w = max(len(x["input_ids"]) for x in b)
    return {"input_ids": torch.tensor([x["input_ids"]+[pad_id]*(w-len(x["input_ids"])) for x in b]),
            "labels":    torch.tensor([x["labels"]   +[-100] *(w-len(x["labels"]))    for x in b]),
            "attention_mask": torch.tensor([[1]*len(x["input_ids"])+[0]*(w-len(x["input_ids"])) for x in b])}

trainer = Trainer(model=model, train_dataset=train_ds, eval_dataset=eval_ds,
    data_collator=collate,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH, gradient_accumulation_steps=ACCUM,
        num_train_epochs=EPOCHS, learning_rate=LR, warmup_steps=5,
        fp16=not BF16, bf16=BF16, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="linear", logging_steps=5,
        # v3 had no eval_dataset, so the train/eval gap - the whole memorization
        # diagnostic at this corpus size - was invisible.
        eval_strategy="epoch", save_strategy="no",
        seed=SEED, output_dir="outputs", report_to=[]))
stats = trainer.train()
LOSSES = [{k: v for k, v in h.items() if k in ("epoch","loss","eval_loss")}
          for h in trainer.state.log_history if "loss" in h or "eval_loss" in h]

In [ ]:
# 11. Eval after, then write the FULL results artifact.
FastLanguageModel.for_inference(model)
model.config.use_cache = True
model.generation_config.max_length = None

RESULTS["tuned"], rows = evaluate(eval_recs, "AFTER FINE-TUNING", "tuned"); PREDICTIONS += rows
RESULTS["train_subset"], rows = evaluate(train_recs[:10], "TRAIN SUBSET (memorization)", "train"); PREDICTIONS += rows

b, t = RESULTS["baseline"], RESULTS["tuned"]
print(f"\nSTRUCTURE  baseline {b['structure']}/{b['n']}  ->  tuned {t['structure']}/{t['n']}")
print(f"FULL       baseline {b['full']}/{b['n']}  ->  tuned {t['full']}/{t['n']}")

# The comparison that actually reads: per decision, baseline vs tuned, with
# chance beside it. Structure can sit at 0/0 while these move a long way.
CHANCE = {"intent": 0.50, "trigger": 0.25, "action": 0.33}
print("\nDECISION            baseline      tuned     delta")
DELTAS = {}
for k in ("intent", "trigger", "action", "n_mutations", "mutation_kinds",
          "col_types", "table_name"):
    db, dt = b.get("decisions", {}).get(k), t.get("decisions", {}).get(k)
    if not (db and dt and db["seen"] and dt["seen"]):
        continue
    rb, rt = db["hit"] / db["seen"], dt["hit"] / dt["seen"]
    DELTAS[k] = {"baseline": rb, "tuned": rt, "delta": rt - rb,
                 "chance": CHANCE.get(k)}
    c = f"  (chance {CHANCE[k]:.0%})" if k in CHANCE else ""
    print(f"  {k:16} {db['hit']:>2}/{db['seen']:<3} {rb:4.0%}  "
          f"{dt['hit']:>2}/{dt['seen']:<3} {rt:4.0%}  {rt-rb:+5.0%}{c}")

os.makedirs(OUT_DIR, exist_ok=True)
ARTIFACTS = {}          # filled by the save cell, re-serialized there
artifact = {
    "run": {"id": RUN_ID, "dir": RUN_DIR, "started_utc": RUN_STARTED,
            "base_model": BASE_MODEL, "seed": SEED, "epochs": EPOCHS,
            "lr": LR, "lora_r": LORA_R, "target": TARGET, "batch": BATCH,
            "accum": ACCUM, "dtype": "bf16" if BF16 else "fp16", "gpu": GPU_NAME,
            "vram_gb": round(VRAM_GB, 1), "trainable_params": TRAINABLE,
            "max_seq_length": MAX_SEQ_LENGTH, "max_new_tokens": MAX_NEW_TOKENS,
            "versions": VERSIONS},
    "corpus": {"path": CORPUS, "sha256_16": CORPUS_SHA, "n": len(records),
               "intents": INTENTS, "train": len(train_recs), "eval": len(eval_recs)},
    "invariants": INVARIANTS,
    "metrics": RESULTS,
    "decision_deltas": DELTAS,
    "losses": LOSSES,
    "predictions": PREDICTIONS,
    "artifacts": ARTIFACTS,
}
with open(f"{OUT_DIR}/results.json", "w", encoding="utf-8") as fh:
    json.dump(artifact, fh, indent=1, ensure_ascii=False)
print(f"\nresults -> {OUT_DIR}/results.json "
      f"({os.path.getsize(f'{OUT_DIR}/results.json')/1024:.0f} KB, "
      f"{len(PREDICTIONS)} predictions, {len(INVARIANTS)} invariants)")

In [ ]:
# 12. Save. Adapter + results + GGUF to Drive, which cell 3 proved writable.
import shutil

model.save_pretrained(OUT_DIR); tokenizer.save_pretrained(OUT_DIR)
ARTIFACTS["adapter_dir"] = OUT_DIR

# GGUF, because the browser target needs one and this is the only place the
# merged weights exist. wllama cannot stage a LoRA adapter at runtime
# (prepareBlobs only ever names files model-NNNNN-of-NNNNN.gguf), so the
# adapter HAS to be baked in - there is no runtime-adapter path to fall back on.
#
# NOTE this merges a 4-bit-trained adapter into a 16-bit base and requantizes,
# which is an approximation rather than an identity. Re-run the eval against the
# GGUF before trusting the numbers above for it.
EXPORT_GGUF = True
GGUF_QUANT  = "q4_k_m"      # what the browser runtime wants; q8_0 for a check

if EXPORT_GGUF:
    # INSIDE OUT_DIR, so the whole run is one directory and one copy.
    gguf_dir = f"{OUT_DIR}/gguf_{GGUF_QUANT}"
    try:
        model.save_pretrained_gguf(gguf_dir, tokenizer,
                                   quantization_method=GGUF_QUANT)
        found = [f"{r}/{f}" for r, _, fs in os.walk(gguf_dir) for f in fs
                 if f.endswith(".gguf")]
        ARTIFACTS["gguf"] = [{"path": f, "bytes": os.path.getsize(f)} for f in found]
        for a in ARTIFACTS["gguf"]:
            print(f"  gguf {a['path']}  {a['bytes']/1024**2:.0f} MiB")
        check("gguf written", bool(found), gguf_dir)

        # DOES THE EMBEDDING TENSOR FIT A PHONE? This decides whether the
        # artifact can run in a browser at all, and the file size does not
        # imply it.
        #
        # Mobile WebGPU reports a 128 MiB maxStorageBufferBindingSize (our
        # Adreno 740 does), and the largest single tensor is token_embd.
        # MEASURED: a stock bartowski Q4_K_M of Qwen2.5-0.5B keeps token_embd
        # at Q8_0 = 137.9 MiB - over the limit, on a model whose whole FILE is
        # 379 MiB. So "q4_k_m" guarantees nothing here; the embedding type has
        # to be checked. If this fails, requantize explicitly:
        #   llama-quantize --token-embedding-type q4_K in.gguf out.gguf Q4_K_M
        BINDING_LIMIT_MIB = 128
        try:
            from gguf import GGUFReader
            for path in found:
                big = sorted(GGUFReader(path).tensors,
                             key=lambda t: -int(t.n_bytes))[:3]
                for t in big:
                    print(f"    {t.name:28} {t.tensor_type.name:8} "
                          f"{int(t.n_bytes)/1024**2:7.1f} MiB")
                worst = int(big[0].n_bytes) / 1024**2
                ARTIFACTS["largest_tensor"] = big[0].name
                ARTIFACTS["largest_tensor_mib"] = round(worst, 1)
                check(f"largest tensor fits the {BINDING_LIMIT_MIB} MiB mobile binding",
                      worst <= BINDING_LIMIT_MIB,
                      f"{big[0].name} is {worst:.1f} MiB - requantize with "
                      "--token-embedding-type q4_K")
        except ImportError:
            print("    (no gguf reader - check token_embd before shipping this "
                  "to a browser)")
    except Exception as e:
        # Do NOT lose the run over an export step - the adapter is the artifact
        # and it is already on disk.
        print(f"  GGUF export failed ({type(e).__name__}: {e})")
        print("  the adapter is unaffected; convert offline with "
              "convert_hf_to_gguf.py + llama-quantize")
        ARTIFACTS["gguf_error"] = f"{type(e).__name__}: {e}"

size = subprocess.run(["du", "-sh", OUT_DIR], capture_output=True, text=True).stdout.split()[0]
print(f"  adapter + results -> {OUT_DIR}  ({size})")

# Re-serialize now that ARTIFACTS is populated - cell 11 wrote it before the
# GGUF existed, and the artifact list is half the point of the file.
artifact["artifacts"] = ARTIFACTS
with open(f"{OUT_DIR}/results.json", "w", encoding="utf-8") as fh:
    json.dump(artifact, fh, indent=1, ensure_ascii=False)

if not os.path.isdir(f"{DRIVE_ROOT}/MyDrive"):
    print("  mount went away mid-run, remounting")
    drive.mount(DRIVE_ROOT, force_remount=True)

# One copy: adapter, tokenizer, results.json and the gguf/ subdirectory all
# live under OUT_DIR. Re-running this cell overwrites THIS run's folder and
# leaves earlier runs alone, because RUN_DIR carries the start timestamp.
os.makedirs(DRIVE_DEST, exist_ok=True)
shutil.copytree(OUT_DIR, DRIVE_DEST, dirs_exist_ok=True)
for root, _, files in os.walk(DRIVE_DEST):
    for f in sorted(files):
        full = os.path.join(root, f)
        print(f"    {os.path.relpath(full, DRIVE_DEST):40} "
              f"{os.path.getsize(full)/1024**2:8.1f} MiB")

landed = os.path.isfile(f"{DRIVE_DEST}/results.json") and \
         os.path.isfile(f"{DRIVE_DEST}/adapter_model.safetensors")
check("adapter and results are on Drive", landed, DRIVE_DEST)
print(f"  reload with FastLanguageModel.from_pretrained({DRIVE_DEST!r})")